[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/03_data_analysis_and_eda/02_eda_workflow.ipynb)

# Bài 02 — Quy trình EDA (Exploratory Data Analysis)

**🎯 Mục tiêu**
- Nắm một **quy trình EDA có hệ thống** thay vì vẽ biểu đồ ngẫu hứng.
- Viết hàm `profile(df)` tóm tắt chất lượng dữ liệu dùng lại cho mọi dự án.
- Phân tích biến mục tiêu và quan hệ feature ↔ target.
- Ghi lại **insight, giả thuyết và rủi ro** (leakage, dữ liệu bẩn) để định hướng bước modeling.

> EDA không phải để "làm đẹp báo cáo". Mục tiêu là **hiểu dữ liệu đủ sâu để không mắc sai lầm khi xây model** — và để có ý tưởng feature tốt.

Dữ liệu: `customer_churn.csv` — bài toán dự đoán khách hàng viễn thông rời bỏ dịch vụ.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/03_data_analysis_and_eda

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()
df = pd.read_csv("../../data/sample/customer_churn.csv")

## Quy trình EDA 7 bước

```text
1. Xác định câu hỏi       → bài toán gì? đơn vị phân tích? biến mục tiêu? metric kinh doanh?
2. Nhìn tổng quan          → shape, kiểu dữ liệu, vài dòng mẫu, data dictionary
3. Chất lượng dữ liệu      → missing, trùng lặp, giá trị bất thường, cardinality
4. Biến mục tiêu           → phân phối, mất cân bằng?
5. Từng biến (univariate)  → phân phối, outlier, lệch
6. Quan hệ (bivariate)     → feature ↔ target, feature ↔ feature
7. Ghi lại                 → insight, giả thuyết, rủi ro leakage, ý tưởng feature
```

## Bước 1 — Xác định câu hỏi

- **Bài toán:** dự đoán khách hàng nào sẽ rời bỏ (churn) để chăm sóc trước → **binary classification**.
- **Đơn vị phân tích:** một khách hàng (một hàng).
- **Biến mục tiêu:** `churn` (1 = đã rời bỏ).
- **Thời điểm dự đoán:** quan trọng! Chỉ được dùng thông tin **có sẵn tại thời điểm dự đoán** (xem bài 04 và chương 05 về leakage).

## Bước 2 — Nhìn tổng quan

In [ ]:
print("shape:", df.shape)
df.head()

In [ ]:
df.dtypes

Viết **data dictionary** — mô tả nghĩa từng cột (trong thực tế hỏi người nắm nghiệp vụ):

| Cột | Ý nghĩa | Kiểu |
|---|---|---|
| `customer_id` | mã khách hàng | định danh (không dùng làm feature) |
| `tenure_months` | số tháng đã dùng dịch vụ | số |
| `contract` | loại hợp đồng | phân loại (3 giá trị) |
| `internet_service` | loại internet | phân loại |
| `monthly_charges` | cước tháng (nghìn VND) | số |
| `total_charges` | tổng cước đã trả | số |
| `payment_method` | phương thức thanh toán | phân loại |
| `support_calls` | số cuộc gọi hỗ trợ | số đếm |
| `senior` | khách hàng cao tuổi | boolean |
| `n_products` | số sản phẩm đang dùng | số đếm |
| `churn` | **mục tiêu**: đã rời bỏ | 0/1 |

## Bước 3 — Chất lượng dữ liệu

Một hàm profile nhỏ — dùng lại cho mọi dataset:

In [ ]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """Tóm tắt nhanh chất lượng dữ liệu theo cột."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
        "pct_unique": (df.nunique() / len(df) * 100).round(2),
        "example": df.iloc[0],
    })


profile(df)

In [ ]:
print("Số hàng trùng lặp hoàn toàn:", df.duplicated().sum())
print("Số customer_id trùng       :", df["customer_id"].duplicated().sum())
print("tenure âm hoặc > 72        :", ((df["tenure_months"] < 0) | (df["tenure_months"] > 72)).sum())
print("cước tháng <= 0            :", (df["monthly_charges"] <= 0).sum())

**Ghi chú:** `customer_id` có 100% giá trị duy nhất → là định danh, **không** dùng làm feature (model sẽ "học thuộc" ID). Nhìn chung dữ liệu này sạch — bài 04 sẽ làm việc với dữ liệu bẩn.

## Bước 4 — Biến mục tiêu

In [ ]:
counts = df["churn"].value_counts().sort_index()
print(counts, "\nTỉ lệ churn:", round(df["churn"].mean(), 3))
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["ở lại (0)", "rời bỏ (1)"], counts.values, color=["#2563eb", "#dc2626"])
ax.set_title("Phân phối biến mục tiêu")
plt.show()

~29% churn → **mất cân bằng nhẹ**. Hệ quả: accuracy không phải metric tốt nhất (model dự đoán "không ai rời bỏ" đã đạt ~71% accuracy!). Nên dùng thêm precision/recall, ROC-AUC (chương 14) và **stratified** split (chương 15).

## Bước 5 — Từng biến

In [ ]:
numeric = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, numeric):
    ax.hist(df[col], bins=30, color="#2563eb", alpha=0.8)
    ax.set_title(f"{col}\nskew = {df[col].skew():.2f}")
plt.show()

In [ ]:
categorical = ["contract", "internet_service", "payment_method", "senior"]
fig, axes = plt.subplots(1, 4, figsize=(20, 3.5))
for ax, col in zip(axes, categorical):
    vc = df[col].value_counts()
    ax.bar(vc.index.astype(str), vc.values)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=20)
plt.show()

**Quan sát:** `total_charges` lệch phải mạnh (skew > 0.9) → có thể cần log-transform cho model tuyến tính; `monthly_charges` có nhiều đỉnh (theo loại internet); hơn một nửa khách hàng dùng hợp đồng tháng.

## Bước 6 — Quan hệ với biến mục tiêu

**Biến phân loại:** so sánh **tỉ lệ churn** giữa các nhóm (kèm số lượng để biết nhóm có đủ lớn không):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
for ax, col in zip(axes, categorical):
    stats_ = df.groupby(col)["churn"].agg(["mean", "size"])
    ax.bar(stats_.index.astype(str), stats_["mean"], color="#dc2626")
    ax.axhline(df["churn"].mean(), color="k", ls="--", lw=1)
    for i, (rate, n) in enumerate(zip(stats_["mean"], stats_["size"])):
        ax.text(i, rate + 0.01, f"n={n}", ha="center", fontsize=8)
    ax.set_title(f"Tỉ lệ churn theo {col}")
    ax.tick_params(axis="x", rotation=20)
plt.show()

**Biến số:** so sánh phân phối giữa hai nhóm churn / không churn:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, numeric):
    data = [df.loc[df["churn"] == k, col] for k in (0, 1)]
    ax.boxplot(data, tick_labels=["ở lại", "rời bỏ"])
    ax.set_title(col)
plt.show()

**Feature ↔ feature:** ma trận tương quan để phát hiện các feature dư thừa (multicollinearity):

In [ ]:
corr = df[numeric + ["churn"]].corr()
print(corr.round(2))

## Bước 7 — Ghi lại insight, giả thuyết và rủi ro

Đây là sản phẩm quan trọng nhất của EDA. Ví dụ ghi chép cho dataset này:

**Insight**
1. Hợp đồng `month-to-month` có tỉ lệ churn cao gấp nhiều lần hợp đồng dài hạn.
2. Khách hàng mới (tenure thấp) rời bỏ nhiều hơn hẳn.
3. Nhiều cuộc gọi hỗ trợ → churn cao (dấu hiệu khách hàng không hài lòng).
4. Internet `fiber` có churn cao hơn `dsl` dù là dịch vụ "tốt hơn" → cần hỏi nghiệp vụ (giá? chất lượng?).

**Giả thuyết cần kiểm chứng:** tương tác `contract × tenure` (khách mới + hợp đồng tháng là nhóm rủi ro nhất).

**Rủi ro / lưu ý cho modeling**
- `total_charges` ≈ `tenure × monthly_charges` (tương quan rất cao với tenure) → thông tin dư thừa; cẩn thận khi diễn giải hệ số model tuyến tính.
- `support_calls`: được đếm **trước** hay **sau** thời điểm dự đoán? Nếu đếm cả các cuộc gọi khi khách hàng gọi để huỷ dịch vụ → **data leakage**!
- Dữ liệu mất cân bằng nhẹ → stratified split, metric phù hợp.
- `customer_id` không dùng làm feature.

**Ý tưởng feature:** `avg_charge = total_charges / tenure`, `is_new_customer = tenure < 6`, nhóm tenure, `calls_per_month`.

## ✅ Checklist EDA

- [ ] Hiểu bài toán, đơn vị phân tích, thời điểm dự đoán
- [ ] Data dictionary cho mọi cột
- [ ] Missing, trùng lặp, giá trị vô lý, cardinality
- [ ] Phân phối biến mục tiêu (mất cân bằng?)
- [ ] Phân phối từng feature (lệch, outlier)
- [ ] Quan hệ feature ↔ target, feature ↔ feature
- [ ] Danh sách insight, giả thuyết, rủi ro leakage, ý tưởng feature

## 🧠 Kiểm tra nhanh

1. Vì sao `customer_id` không nên dùng làm feature?
2. Model luôn dự đoán "không churn" đạt accuracy bao nhiêu trên dữ liệu này?
3. Cho ví dụ một feature có thể gây data leakage trong bài toán churn.

## 📚 Tài liệu tham khảo

- John Tukey — *Exploratory Data Analysis* (1977) — người đặt ra thuật ngữ EDA
- Google — *Rules of ML*, phần "Before Machine Learning": https://developers.google.com/machine-learning/guides/rules-of-ml